# Day 1: Hydrogen-Induced Defect Transport

This notebook creates the first physics-informed simulation pipeline for a one-dimensional lattice with hydrogen-like local defects.


## Mathematical Formulas Used in This Notebook

This notebook is based on a simple one-dimensional tight-binding model. The lattice has $N$ sites, and the quantum state is written as

$$
|\psi_n\rangle = \sum_{i=1}^{N} c_i^{(n)} |i\rangle,
$$

where $c_i^{(n)}$ is the amplitude of eigenstate $n$ on lattice site $i$.

The Hamiltonian matrix is

$$
H_{ij} = t(\delta_{i,j+1}+\delta_{i,j-1}) + \sum_{d \in D} V_d\,\delta_{i,d}\delta_{j,d},
$$

where $t$ is the nearest-neighbor hopping parameter, $D$ is the set of defect positions, and $V_d$ is the defect strength.

The energy eigenvalue problem is

$$
H\mathbf{c}^{(n)} = E_n\mathbf{c}^{(n)}.
$$

The inverse participation ratio, which measures localization, is

$$
\mathrm{IPR}_n = \sum_{i=1}^{N} |c_i^{(n)}|^4.
$$

A low IPR means that the eigenstate is spread over many sites. A high IPR means that the eigenstate is localized around only a few sites.

The simple transport descriptor used here is

$$
T = \frac{1}{1 + \langle \mathrm{IPR} \rangle},
$$

where

$$
\langle \mathrm{IPR} \rangle = \frac{1}{N}\sum_{n=1}^{N}\mathrm{IPR}_n.
$$

So, when localization increases, $\langle \mathrm{IPR} \rangle$ increases and the transport descriptor $T$ decreases.


## How to read this notebook

This notebook is written as a first, gentle introduction to the physics pipeline.  
You should read each code cell in this order:

1. **Define the computational environment**: import libraries and create folders.
2. **Build the Hamiltonian**: represent a one-dimensional material as a matrix.
3. **Add hydrogen-like defects**: model local perturbations on selected lattice sites.
4. **Solve the eigenvalue problem**: obtain energy states and wave functions.
5. **Measure localization**: use the inverse participation ratio (IPR).
6. **Create a small dataset**: repeat the simulation for many random defect settings.

The most important idea is that hydrogen-like defects modify the Hamiltonian, and those changes can be translated into measurable descriptors such as energy bandwidth, localization, and a simple transport proxy.


## Scientific Pipeline

Hydrogen defects -> lattice Hamiltonian -> energy spectrum -> localization -> transport descriptor -> ML/QML dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
FIGURE_DIR = ROOT / 'figures'

DATA_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

rng = np.random.default_rng(42)

### Explanation of the code cell above

This cell prepares the notebook environment.

- `Path` is used to create operating-system-independent file paths.
- `matplotlib.pyplot` is used for plotting spectra, localization, and dataset trends.
- `numpy` is used for matrix operations and eigenvalue calculations.
- `pandas` is used to store simulation results as a table.
- `DATA_DIR` and `FIGURE_DIR` are created so that generated datasets and plots are saved automatically.
- `rng = np.random.default_rng(42)` creates a reproducible random-number generator. The fixed seed means that the same random defect configurations will be generated every time the notebook is run.

This cell does not perform physics yet. It only prepares the tools and folders needed for the rest of the notebook.


## Model Functions

In [ ]:
def build_hamiltonian(n_sites=40, hopping=-1.0, defect_positions=None, defect_strength=0.0):
    """Build a 1D tight-binding Hamiltonian with optional local defects."""
    hamiltonian = np.zeros((n_sites, n_sites), dtype=float)

    for i in range(n_sites - 1):
        hamiltonian[i, i + 1] = hopping
        hamiltonian[i + 1, i] = hopping

    if defect_positions is not None:
        for position in defect_positions:
            hamiltonian[position, position] += defect_strength

    return hamiltonian


def inverse_participation_ratio(eigenvectors):
    """Compute IPR for each eigenvector. Higher IPR means stronger localization."""
    return np.sum(np.abs(eigenvectors) ** 4, axis=0)


def transport_descriptor(ipr_values):
    """Simple proxy: lower mean localization suggests easier transport."""
    return 1.0 / (1.0 + np.mean(ipr_values))

### Mathematical meaning of this code cell

The function `build_hamiltonian` constructs the matrix form of

$$
H_{ij}=t(\delta_{i,j+1}+\delta_{i,j-1})+\sum_{d\in D}V_d\delta_{i,d}\delta_{j,d}.
$$

The function `inverse_participation_ratio` computes

$$
\mathrm{IPR}_n=\sum_i |c_i^{(n)}|^4.
$$

The function `transport_descriptor` computes

$$
T=\frac{1}{1+\langle \mathrm{IPR}\rangle}.
$$


### Explanation of the code cell above

This cell defines the core physics functions.

`build_hamiltonian` creates a one-dimensional tight-binding Hamiltonian. In this simplified model:

- each lattice site is represented by one row and one column of the matrix,
- neighboring sites are connected by the hopping term,
- hydrogen-like defects are modeled as local diagonal perturbations.

`inverse_participation_ratio` computes the IPR of each eigenvector. IPR is a localization measure:

- low IPR means the state is spread over many lattice sites,
- high IPR means the state is concentrated on fewer sites.

`transport_descriptor` is a simple proxy for transport. It is larger when the average localization is smaller. Therefore, in this toy model, stronger delocalization corresponds to easier transport.


## Single Example

In [ ]:
n_sites = 40
defect_positions = [10, 24, 31]
defect_strength = 2.5

H = build_hamiltonian(
    n_sites=n_sites,
    defect_positions=defect_positions,
    defect_strength=defect_strength,
)

eigenvalues, eigenvectors = np.linalg.eigh(H)
ipr_values = inverse_participation_ratio(eigenvectors)
most_localized_index = int(np.argmax(ipr_values))

print(f'Mean IPR: {np.mean(ipr_values):.4f}')
print(f'Max IPR: {np.max(ipr_values):.4f}')
print(f'Transport descriptor: {transport_descriptor(ipr_values):.4f}')

### Mathematical meaning of this code cell

The function `build_hamiltonian` constructs the matrix form of

$$
H_{ij}=t(\delta_{i,j+1}+\delta_{i,j-1})+\sum_{d\in D}V_d\delta_{i,d}\delta_{j,d}.
$$

The function `inverse_participation_ratio` computes

$$
\mathrm{IPR}_n=\sum_i |c_i^{(n)}|^4.
$$

The function `transport_descriptor` computes

$$
T=\frac{1}{1+\langle \mathrm{IPR}\rangle}.
$$


### Explanation of the code cell above

This cell runs the first complete simulation.

It chooses:

- a lattice with `40` sites,
- three hydrogen-like defect positions: `10`, `24`, and `31`,
- a defect strength of `2.5`.

Then it builds the Hamiltonian, solves the eigenvalue problem using `np.linalg.eigh`, and computes the IPR values of the eigenvectors.

The printed values summarize the physical behavior:

- **Mean IPR**: average localization over all eigenstates.
- **Max IPR**: the strongest localization observed in the system.
- **Transport descriptor**: a compact score where larger values suggest more delocalized, more transport-friendly behavior.

This is the first place where the abstract Hamiltonian becomes measurable numerical descriptors.


## Day 1 Figures

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(eigenvalues, marker='o', linewidth=1)
plt.xlabel('State index')
plt.ylabel('Energy')
plt.title('Energy spectrum with hydrogen-like defects')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'energy_spectrum_day1.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(ipr_values, marker='o', linewidth=1, color='tab:red')
plt.xlabel('State index')
plt.ylabel('IPR')
plt.title('Inverse participation ratio')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ipr_values_day1.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(np.abs(eigenvectors[:, most_localized_index]) ** 2, marker='o')
plt.xlabel('Lattice site')
plt.ylabel('Probability')
plt.title('Most localized eigenstate')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'most_localized_state_day1.png', dpi=200)
plt.show()

### Explanation of the code cell above

This cell visualizes three important outputs from the simulation.

1. **Energy spectrum**  
   The first plot shows the eigenvalues of the defected Hamiltonian. These values represent the allowed energy states of the lattice model.

2. **IPR values**  
   The second plot shows localization for each eigenstate. Peaks in this plot indicate eigenstates that are more localized.

3. **Most localized eigenstate**  
   The third plot shows the spatial probability distribution of the eigenstate with the highest IPR. This helps you see where localization occurs on the lattice.

These plots are useful because they connect matrix calculations to physical interpretation.


## Generate the First Dataset

In [ ]:
records = []

for sample_id in range(200):
    n_defects = int(rng.integers(1, 7))
    positions = sorted(rng.choice(n_sites, size=n_defects, replace=False).tolist())
    strength = float(rng.uniform(0.1, 4.0))

    H_sample = build_hamiltonian(
        n_sites=n_sites,
        defect_positions=positions,
        defect_strength=strength,
    )
    values, vectors = np.linalg.eigh(H_sample)
    ipr = inverse_participation_ratio(vectors)

    records.append(
        {
            'sample_id': sample_id,
            'n_sites': n_sites,
            'n_defects': n_defects,
            'defect_positions': ';'.join(map(str, positions)),
            'defect_strength': strength,
            'mean_energy': float(np.mean(values)),
            'energy_bandwidth': float(np.max(values) - np.min(values)),
            'mean_ipr': float(np.mean(ipr)),
            'max_ipr': float(np.max(ipr)),
            'transport_descriptor': float(transport_descriptor(ipr)),
        }
    )

dataset = pd.DataFrame(records)
dataset.head()

### Mathematical meaning of this code cell

The function `build_hamiltonian` constructs the matrix form of

$$
H_{ij}=t(\delta_{i,j+1}+\delta_{i,j-1})+\sum_{d\in D}V_d\delta_{i,d}\delta_{j,d}.
$$

The function `inverse_participation_ratio` computes

$$
\mathrm{IPR}_n=\sum_i |c_i^{(n)}|^4.
$$

The function `transport_descriptor` computes

$$
T=\frac{1}{1+\langle \mathrm{IPR}\rangle}.
$$


### Mathematical meaning of this dataset-generation cell

For every random sample, the notebook solves

$$
H\mathbf{c}^{(n)} = E_n\mathbf{c}^{(n)}
$$

and stores numerical descriptors such as

$$
B = E_{\max}-E_{\min}, \qquad \overline{\mathrm{IPR}} = \frac{1}{N}\sum_n \mathrm{IPR}_n.
$$

These descriptors become one row of the dataset.


### Explanation of the code cell above

This cell creates a small simulation dataset.

Instead of studying only one defect configuration, the code generates `200` random configurations. For each sample, it randomly chooses:

- the number of defects,
- the defect positions,
- the defect strength.

For every configuration, the code builds a Hamiltonian, solves its eigenvalues and eigenvectors, computes localization descriptors, and stores the result in a dictionary.

Finally, all dictionaries are converted into a `pandas` DataFrame called `dataset`.

Each row of this dataset is one simulated material configuration. This is the first step toward using machine learning, because ML models need structured input-output examples.


In [ ]:
dataset_path = DATA_DIR / 'week1_hydrogen_defect_dataset.csv'
dataset.to_csv(dataset_path, index=False)
dataset_path

### Explanation of the code cell above

This cell saves the generated dataset as a CSV file.

The file is saved inside the `data` directory with the name:

`week1_hydrogen_defect_dataset.csv`

Saving the dataset is important because later notebooks can load the same data without rerunning every simulation. This also makes the workflow more reproducible.


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(dataset['n_defects'], dataset['mean_ipr'], alpha=0.75)
plt.xlabel('Number of defects')
plt.ylabel('Mean IPR')
plt.title('Defects vs localization')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'defects_vs_ipr_day1.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(dataset['defect_strength'], dataset['transport_descriptor'], alpha=0.75, color='tab:green')
plt.xlabel('Defect strength')
plt.ylabel('Transport descriptor')
plt.title('Defect strength vs transport descriptor')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'defect_strength_vs_transport_day1.png', dpi=200)
plt.show()

### Explanation of the code cell above

This cell creates exploratory plots from the generated dataset.

The first scatter plot compares the number of defects with the mean IPR. It helps answer:

> Do more defects generally produce stronger localization?

The second scatter plot compares defect strength with the transport descriptor. It helps answer:

> Do stronger defects reduce or increase the transport proxy?

These plots do not prove causality, but they give an early visual understanding of how defect parameters may influence localization and transport.


## Day 1 Conclusion

Today, we built the first physics-informed lattice model and generated the first dataset for hydrogen-defect transport analysis.

## Key learning takeaway

After running this notebook, focus on writing one or two sentences in your own words:

- What physical quantity was computed?
- Which feature or parameter changed?
- How did the result affect localization or transport?

This habit will help you turn code execution into scientific understanding.


## Final Formula Reminder

The core logic of this notebook can be summarized as

$$
\text{defect configuration} \rightarrow H \rightarrow \{E_n,\mathbf{c}^{(n)}\} \rightarrow \text{features} \rightarrow \text{prediction or optimization}.
$$

This is the main physics-informed workflow: we start from a material-inspired defect model, convert it into spectral and localization descriptors, and then use those descriptors for machine learning, interpretation, or optimization.
